# 스내치 측면 영상에서 네 관절 확인하기

기존 사람별 검증 영상에서 YOLO26n-pose가 카메라에 가까운 쪽의 어깨·골반·무릎·발목을 어디에 찍는지 확인합니다. 왼쪽에는 원본 전체, 오른쪽에는 같은 프레임에 관절을 겹쳐 봅니다. 원판 검출은 기존 `plate_baseline/weights/best.pt`를 유지하며 이 노트북은 자세 모델만 실행합니다.

### 1. 검증 영상과 이벤트 찾기

표의 숫자는 0부터 세는 원본 프레임 번호입니다. 빈 이벤트는 아직 판독한 시점이 없다는 뜻입니다. `near_side`는 카메라에 가까운 쪽으로 기록된 **선수 본인의 좌우**입니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
from ultralytics import YOLO

from wlcv.annotation.storage import EVENTS, load_events
from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.plots import predict_rgb
from wlcv.annotation.pose_points import load_points, pose_point_path
from wlcv.evaluation.pose import compare_manual_points, joint_table, plot_pose, show_candidates

split = pd.read_csv(SPLIT)
validation = split.loc[split["role"] == "validation", ["video_id", "person_id"]]
metadata = pd.read_csv(ROOT / "data/metadata/videos.csv")
videos = validation.merge(metadata[["video_id", "filename", "frame_count", "near_side"]],
                          on="video_id", validate="one_to_one", how="left").set_index("video_id")
events = {video: load_events(ANNOTATIONS, video, pass_number=1) for video in videos.index}
event_frames = pd.DataFrame.from_dict(events, orient="index").reindex(columns=EVENTS).astype("Int64")
display(videos[["person_id", "frame_count", "near_side"]].join(event_frames).style.format(na_rep="—"))

### 2. 동작 중 프레임 고르기

아래 세 장은 서로 다른 선수의 무릎 통과, 바와 몸의 접촉, 캐치입니다. 다른 장면을 보려면 목록의 영상 ID와 이벤트를 바꿉니다. 프레임 번호를 직접 넣을 수도 있습니다. 예를 들어 `("a03_sn_f_01", 20)`은 20번 프레임을 읽습니다.

In [ ]:
samples = [
    ("a03_sn_f_01", "transition"),
    ("b02_sn_h_02", "drive"),
    ("c04_sn_f_01", "catch"),
]

# 첫 실행 때 공식 가중치를 models/에 내려받고 이후에는 로컬 파일을 사용한다.
pose_weights = ROOT / "models/yolo26n-pose.pt"
pose_weights.parent.mkdir(exist_ok=True)
model = YOLO(str(pose_weights))
if model.task != "pose":
    raise ValueError("yolo26n-pose.pt 자세 추정 가중치를 확인하세요.")

### 3. 원본에서 사람 후보 찾기

CPU에서 한 장씩 읽습니다. 사람 박스 기준은 0.25, 입력 크기는 640입니다. 사람이 여러 명이면 사진에 붙은 번호로 실제 수행자를 확인합니다. 사람이 한 명이면 바로 다음 단계에서 그 사람의 관절을 봅니다.

In [ ]:
frames = {}
overview = []
for video, point in tqdm(samples, desc="검증 프레임", unit="장"):
    if video not in videos.index:
        raise ValueError(f"{video}가 첫 표의 검증 영상인지 확인하세요.")
    if isinstance(point, str):
        if point not in events[video]:
            raise ValueError(f"{video}에는 {point} 라벨이 없습니다. 프레임 번호를 넣으세요.")
        index = events[video][point]
    else:
        index = point
    if not isinstance(index, int) or not 0 <= index < videos.at[video, "frame_count"]:
        raise ValueError(f"{video}의 프레임 번호 범위를 확인하세요.")

    rgb = read_rgb(VIDEOS / videos.at[video, "filename"], index)
    result = predict_rgb(model, rgb, conf=0.25, imgsz=640, device="cpu").cpu().numpy()
    frames[video, index] = (rgb, result)
    count = len(result.boxes)
    overview.append((video, point, index, count, videos.at[video, "near_side"]))
    if count != 1:
        show_candidates(rgb, result, f"{video} / frame {index}")
        plt.show()

display(pd.DataFrame(overview, columns=["영상", "선택한 시점", "프레임", "사람 후보 수", "가까운 쪽"]))

### 4. 원본과 관절 표시본 비교하기

그림과 표에는 영상의 `near_side`에 해당하는 네 관절이 나옵니다. 좌우가 뒤바뀌었는지 확인할 때만 `show_both_sides = True`로 바꾸세요.

청록색 원은 **선수의 왼쪽**, 주황색 삼각형은 **오른쪽**입니다. 어깨 → 골반 → 무릎 → 발목 순으로 연결합니다. 빈 표식은 신뢰도가 기준보다 낮은 관절입니다. 이 점은 선에서 제외하고 원래 점수는 표에 남깁니다. 표의 좌표는 원본 왼쪽 위가 (0, 0)인 픽셀값입니다.

여러 후보가 나오면 `person_choices`에 **(영상 ID, 실제 프레임 번호)**와 사람 번호를 넣고 아래 셀만 다시 실행하세요. 후보 번호는 프레임마다 새로 매깁니다. 유일한 후보가 배경 인물이라면 그 프레임에 `None`을 지정해 건너뜁니다.

[수동 관절 판독 노트북](../annotation/03_pose_points.ipynb)에서 저장한 프레임은 아래 셀에서 판독 결과도 함께 읽습니다. Δx와 Δy는 수동 위치에서 예측 위치를 뺀 값입니다. Δx가 양수면 수동 점이 화면 오른쪽에 있습니다. 사람이 위치를 확신하지 못했거나 예측 좌표가 없으면 거리를 비워 둡니다.

수동 좌표가 있으면 자홍색 십자(일반 클릭점)와 빈 마름모(가림 또는 소켓 위치 추정점)가 예측 위에 표시됩니다. 추정점과 모델의 거리는 표의 **추정점 참고 거리**에만 기록합니다. 직접 확인했다고 기록한 점의 위치 오차와 구분합니다.

In [ ]:
person_choices = {}  # 예: {("a03_sn_f_01", 19): 1}
min_joint_confidence = 0.50  # 빈 표식으로 그릴지 정하는 시각화 기준
show_both_sides = False     # True이면 반대쪽 관절도 참고용으로 표시한다
zoom = True                 # False로 바꾸면 두 그림을 같은 배율로 본다

for (video, index), (rgb, result) in frames.items():
    count = len(result.boxes)
    number = person_choices.get((video, index), 1 if count == 1 else None)
    if count == 0:
        print(f"{video} {index}번: 검출된 사람이 없습니다.")
        continue
    if number is None:
        print(f"{video} {index}번: 수행자 선택을 보류했습니다.")
        continue

    joints = joint_table(result, number, min_joint_confidence)
    near_side = videos.at[video, "near_side"]
    if not show_both_sides:
        if near_side not in ("left", "right"):
            print(f"{video} {index}번: 가까운 쪽이 미확인이라 관절 판독을 보류합니다.")
            continue
        joints = joints.loc[joints["방향"] == near_side]

    manual = load_points(pose_point_path(ANNOTATIONS, video, index))
    fig = plot_pose(rgb, result, number, joints, f"{video} / frame {index}", near_side,
                    zoom=zoom, manual=manual)
    plt.show()
    print(f"{video} {index}번 · 후보 {number} · 사람 박스 신뢰도 {result.boxes.conf[number-1]:.3f}")
    display(joints.style.format({"x (px)": "{:.1f}", "y (px)": "{:.1f}",
                                 "관절 신뢰도": "{:.3f}"}, na_rep="—"))

    if manual:
        comparison = compare_manual_points(joints, manual, near_side)
        display(comparison.style.format(precision=1, na_rep="—"))

### 첫 세 프레임에서 확인한 점

- **a03 19번 · 왼쪽 골반**은 점의 신뢰도가 0.940이지만 실제 골반 위치와 어긋나 보였습니다.
- **b02 36번 · 오른쪽 어깨**는 점의 신뢰도가 0.967입니다. 원판이 겹쳐 정확한 어깨 위치 판독을 보류했습니다.
- **c04 78번 · 왼쪽 어깨**는 점의 신뢰도가 0.995입니다. 트레이너는 기준점이 현재 표시보다 화면 오른쪽에 조금 더 가깝다고 봤습니다. 정확한 픽셀 차이는 아직 측정하지 않았습니다.

가까운 쪽에서도 사람이 확인할 수 없는 관절은 ‘판독 불가’로 남깁니다. 위치 오차를 잴 때는 예측을 숨긴 원본에서 찍은 수동 좌표를 사용합니다. 반대쪽 점은 좌우 혼동을 점검하는 참고 자료입니다. [Ultralytics Pose의 관절 번호와 출력 형식](https://docs.ultralytics.com/tasks/pose/#keypoint-index-map)